# 07 — Retries, Attempts and Idempotency

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
How can one logical operation produce multiple physical task attempts, and why does an arbitrary sink need idempotency?

**Why this matters:** Spark retries failed tasks and may speculate slow ones. Commit coordination for selected output paths is not exactly-once for every HTTP call or custom side effect.

In [ ]:
from pyspark.sql import SparkSession
from pyspark import TaskContext
import os, sys, tempfile, pathlib
os.environ['PYSPARK_PYTHON'] = sys.executable
spark=(SparkSession.builder.master('local[2,2]').appName('thread-lab-07')
       .config('spark.speculation','false').getOrCreate())
sc=spark.sparkContext
attempt_log=pathlib.Path(tempfile.mkdtemp(prefix='spark-retry-lab-'))/'attempts.log'
def fail_once(rows):
    ctx=TaskContext.get()
    attempt=ctx.attemptNumber()
    partition=ctx.partitionId()
    # Disposable local file stands in for a non-transactional append-only sink.
    with attempt_log.open('a',encoding='utf-8') as log:
        log.write(f'operation=demo-42 partition={partition} attempt={attempt}\n')
    if attempt == 0:
        raise RuntimeError('Intentional first-attempt failure for the lab')
    for row in rows:
        yield (row,partition,attempt)
result=(sc.parallelize(['demo-42'],1).mapPartitions(fail_once).collect())
print('Successful logical result:',result)
print('Physical sink attempts:')
print(attempt_log.read_text(encoding='utf-8'))
print('Distinct logical operations:',len({line.split()[0] for line in attempt_log.read_text(encoding='utf-8').splitlines()}))

In [ ]:
print('The append-only attempt log contains one row per physical attempt, even though Spark returned one logical result.')

## Extend safely
In an isolated cluster, you may deliberately fail one test task and observe `TaskContext.attemptNumber()`. Keep writes in-memory or in a disposable local table. Do not use email, payments, shared storage or production endpoints.

**Expected:** attempt numbers are metadata for execution attempts; duplicate side effects remain possible unless the sink transaction or idempotency key absorbs them.

**Why:** task retry is a computation guarantee, not a universal transaction protocol. Stage attempt, partition, task attempt and sink transaction context all matter.

In [ ]:
spark.stop()